In [0]:
# Databricks notebook source

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

bronze_table = "workspace.bronze.yellow_taxi_trips"
silver_table = "workspace.silver.yellow_taxi_trips"

# COMMAND ----------

df_bronze = spark.table(bronze_table)

# COMMAND ----------

df_silver = (
    df_bronze
    .select(
        F.col("VendorID").cast("int").alias("vendor_id"),

        F.col("passenger_count")
            .cast("double")
            .alias("passenger_count"),

        F.col("total_amount")
            .cast("double")
            .alias("total_amount"),

        F.to_timestamp(
            F.col("tpep_pickup_datetime")
        ).alias("pickup_datetime"),

        F.to_timestamp(
            F.col("tpep_dropoff_datetime")
        ).alias("dropoff_datetime")
    )

    .filter(F.col("vendor_id").isNotNull())
    .filter(F.col("passenger_count").isNotNull())
    .filter(F.col("total_amount").isNotNull())
    .filter(F.col("pickup_datetime").isNotNull())
    .filter(F.col("dropoff_datetime").isNotNull())

    .filter(F.col("passenger_count") >= 0)
    .filter(F.col("total_amount") >= 0)

    .withColumn(
        "pickup_year",
        F.year("pickup_datetime")
    )

    .withColumn(
        "pickup_month",
        F.month("pickup_datetime")
    )

    .withColumn(
        "pickup_day",
        F.dayofmonth("pickup_datetime")
    )

    .withColumn(
        "pickup_hour",
        F.hour("pickup_datetime")
    )
    .filter(F.col("pickup_year") == 2023)
    .filter(F.col("pickup_month").between(1, 5))
)

# COMMAND ----------

display(df_silver.limit(10))

# COMMAND ----------

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy(
        "pickup_year",
        "pickup_month"
    )
    .saveAsTable(silver_table)
)

# COMMAND ----------

display(
    spark.table(silver_table)
)

# COMMAND ----------

spark.sql(f"""
SELECT
    pickup_year,
    pickup_month,
    COUNT(*) AS total_records
FROM {silver_table}
GROUP BY
    pickup_year,
    pickup_month
ORDER BY
    pickup_month
""").show()